# Exploratory Data Analysis - Email/SMS Spam Detection

This notebook explores the SMS Spam Collection dataset to understand class distributions, message length characteristics, and key term differences between spam and ham (legitimate) messages.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import sys

# Add src directory to system path to import local preprocessing
sys.path.append('../src')
from data_preprocessing import load_data, preprocess_text

## 1. Load and Inspect Dataset

In [ ]:
# Load dataset
df = load_data('../dataset/spam.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Check for missing values and data types
df.info()

## 2. Label Distribution Analysis

In [ ]:
# Print counts and percentages
counts = df['label'].value_counts()
percentages = df['label'].value_counts(normalize=True) * 100

for label in counts.index:
    print(f"{label.upper()}: {counts[label]} messages ({percentages[label]:.2f}%)")

In [ ]:
# Plot label distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, palette='Set2')
plt.title('Distribution of Ham vs Spam Messages')
plt.xlabel('Category')
plt.ylabel('Count')

os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Message Length Analysis

In [ ]:
# Calculate character and word count
df['char_count'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

# Statistical summary for char and word count by class
df.groupby('label')[['char_count', 'word_count']].describe().transpose()

In [ ]:
# Visualize message length distribution
plt.figure(figsize=(12, 5))

# Character count distribution
plt.subplot(1, 2, 1)
sns.histplot(data=df, x='char_count', hue='label', bins=50, kde=True, palette='Set1', multiple='stack')
plt.xlim(0, 300)  # Zoom in on main message length region
plt.title('Character Count Distribution')
plt.xlabel('Character Count')
plt.ylabel('Frequency')

# Word count distribution
plt.subplot(1, 2, 2)
sns.histplot(data=df, x='word_count', hue='label', bins=50, kde=True, palette='Set1', multiple='stack')
plt.xlim(0, 70)   # Zoom in on main word count region
plt.title('Word Count Distribution')
plt.xlabel('Word Count')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

## 4. Word Clouds for Spam and Ham

In [ ]:
print("Preprocessing text (this may take a few seconds)...")
df['cleaned_message'] = df['message'].apply(preprocess_text)

spam_words = ' '.join(df[df['label'] == 'spam']['cleaned_message'])
ham_words = ' '.join(df[df['label'] == 'ham']['cleaned_message'])

print("Generating word clouds...")

In [ ]:
# Wordcloud for Spam
spam_wc = WordCloud(width=800, height=400, background_color='black', colormap='Reds', random_state=42).generate(spam_words)

plt.figure(figsize=(10, 6))
plt.imshow(spam_wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Spam Messages', fontsize=16, pad=15)
plt.savefig('../outputs/wordcloud.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Wordcloud for Ham
ham_wc = WordCloud(width=800, height=400, background_color='black', colormap='Greens', random_state=42).generate(ham_words)

plt.figure(figsize=(10, 6))
plt.imshow(ham_wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Ham Messages', fontsize=16, pad=15)
plt.show()